In [ ]:
import json

from mxdataspark import MXData


In [ ]:
from importlib.metadata import version
print(f'MXData Spark package version: {version("mxdataspark")}')


In [ ]:
# DBTITLE 1,Widget Parameters
model = dbutils.widgets.text("model", "")
model_entity_type = dbutils.widgets.text("model_entity_type", "")
model_config = dbutils.widgets.text("model_config", "")  # columns
target_entity_schema = dbutils.widgets.text("target_entity_schema", "")
target_entity_name = dbutils.widgets.text("target_entity_name", "")
source_entity_schema = dbutils.widgets.text("source_entity_schema", "")
source_entity_name = dbutils.widgets.text("source_entity_name", "")
source_notebook = dbutils.widgets.text("source_notebook", "")
catalog_name = dbutils.widgets.text("catalog_name", "", "catalog_name")


In [ ]:
model = dbutils.widgets.get("model")
model_entity_type = dbutils.widgets.get("model_entity_type")
data_model_config = json.loads(dbutils.widgets.get("model_config"))  # columns
source_table = f"{dbutils.widgets.get('source_entity_schema')}.{dbutils.widgets.get('source_entity_name')}".lower()
object_schema = dbutils.widgets.get("target_entity_schema").lower()
object_name = dbutils.widgets.get("target_entity_name").lower()
target_table = f"{object_schema}.{object_name}"
source_notebook = dbutils.widgets.get("source_notebook")

catalog_name = dbutils.widgets.get("catalog_name").lower()
spark.catalog.setCurrentCatalog(catalog_name)


In [ ]:
# DBTITLE 1,Stage
try:
    dbutils.notebook.run(
        source_notebook,
        600,
        {
            "dimension_key": 0,
            "catalog_name": catalog_name,
            "source_table": source_table,
        },
    )
except Exception as e:
    dbutils.notebook.exit(
        json.dumps(
            {
                "modelSchema": source_entity_schema,
                "modelObject": source_entity_name,
                "errors": f"Stage build failure on notebook: {source_notebook}",
                "success": 0,
            }
        )
    )


In [ ]:
# DBTITLE 1,Main
dbutils.notebook.exit(
    MXData.model(model_entity_type)
    .source_data(source_table)
    .target_table(target_table)
    .target_columns(data_model_config)
    .model()
    .metrics()
)
